<a href="https://colab.research.google.com/github/kavyasree4494/OIBSIP/blob/main/OIBSIP_DataAnalytics_L3_Cleaning_Data_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
file_path = '/content/data quality report.csv'
try:
    df = pd.read_csv(file_path)
    print(f"Successfully loaded dataset: {file_path}")
    print(f"Shape: {df.shape[0]} rows, {df.shape[1]} columns\n")
    display(df.head())
except Exception as e:
    print(f"Error loading {file_path}: {e}")
    df = pd.DataFrame({
        'id': [1, 2, 2, 4, 5],
        'age': [25, 30, 30, -5, 200],
        'score': [85.5, np.nan, 85.5, 90.0, 95.0],
        'date': ['2023-01-01', '2023-01-02', '2023-01-02', 'invalid_date', '2023-01-05']
    })
    print("Running report on fallback/sample data instead:\n")
    display(df)
print("DATA QUALITY REPORT")
print("NULL VALUE COUNT PER COLUMN:")
nulls = df.isnull().sum()
null_pct = (df.isnull().sum() / len(df)) * 100
null_report = pd.DataFrame({'Null Count': nulls, 'Percentage (%)': null_pct.round(2)})
display(null_report)
print(" DUPLICATE ROWS:")
duplicate_count = df.duplicated().sum()
print(f"Total exact duplicate rows: {duplicate_count}")
if duplicate_count > 0:
    print("Displaying duplicates (including original occurrences):")
    display(df[df.duplicated(keep=False)].head(10))
print("COLUMN DATA TYPES:")
display(df.dtypes.to_frame(name='Data Type'))
print("VALUE RANGE & ANOMALIES:")
numeric_cols = df.select_dtypes(include=[np.number]).columns
if len(numeric_cols) > 0:
    range_df = df[numeric_cols].agg(['min', 'max', 'mean', 'std']).T
    print("Summary statistics for numeric columns:")
    display(range_df)
    for col in numeric_cols:
        col_clean = df[col].dropna()
        if len(col_clean) > 0 and col_clean.std() > 0:
            z_scores = (df[col] - df[col].mean()) / df[col].std()
            outliers = df[z_scores.abs() > 3]
            if not outliers.empty:
                print(f"Potential statistical outliers in '{col}' (Z-score > 3):")
                display(outliers[[col]])
else:
    print("No numeric columns found for range analysis.")

Successfully loaded dataset: /content/data quality report.csv
Shape: 20 rows, 11 columns



,Rank,Peak,All Time Peak,Actual gross,Adjusted gross (in 2022 dollars),Artist,Tour title,Year(s),Shows,Average gross,Ref.
0,1,1,2,"$780,000,000","$780,000,000",Taylor Swift,The Eras Tour †,2023–2024,56,"$13,928,571",[1]
1,2,1,7[2],"$579,800,000","$579,800,000",Beyoncé,Renaissance World Tour,2023,56,"$10,353,571",[3]
2,3,1[4],2[5],"$411,000,000","$560,622,615",Madonna,Sticky & Sweet Tour ‡[4][a],2008–2009,85,"$4,835,294",[6]
3,4,2[7],10[7],"$397,300,000","$454,751,555",Pink,Beautiful Trauma World Tour,2018–2019,156,"$2,546,795",[7]
4,5,2[4],NaN,"$345,675,146","$402,844,849",Taylor Swift,Reputation Stadium Tour,2018,53,"$6,522,173",[8]


DATA QUALITY REPORT
NULL VALUE COUNT PER COLUMN:


,Null Count,Percentage (%)
Rank,0,0.0
Peak,11,55.0
All Time Peak,14,70.0
Actual gross,0,0.0
Adjusted gross (in 2022 dollars),0,0.0
Artist,0,0.0
Tour title,0,0.0
Year(s),0,0.0
Shows,0,0.0
Average gross,0,0.0


 DUPLICATE ROWS:
Total exact duplicate rows: 0
COLUMN DATA TYPES:


,Data Type
Rank,int64
Peak,object
All Time Peak,object
Actual gross,object
Adjusted gross (in 2022 dollars),object
Artist,object
Tour title,object
Year(s),object
Shows,int64
Average gross,object


VALUE RANGE & ANOMALIES:
Summary statistics for numeric columns:


,min,max,mean,std
Rank,1.0,20.0,10.45,5.942488
Shows,41.0,325.0,110.00,66.507617


Potential statistical outliers in 'Shows' (Z-score > 3):


,Shows
13,325


### Strategy and Justification for Handling Missing Data

Based on our data quality report, the dataset has missing values in the following columns:
- **`Peak`** (55.0% missing)
- **`All Time Peak`** (70.0% missing)

#### Justifications:
1. **`Peak` & `All Time Peak`** (Categorical/Ordinal Numeric containing strings like `1[4]`):
   - **Why are they missing?** The missing values in these peak positions likely represent instances where the tour did not reach a historical peak milestone (e.g., it wasn't the #1 all-time grossing tour at any point, or the peak chart data was not recorded/applicable).
   - **Strategy Chosen:** **Placeholder Imputation / "Not Applicable"**. Due to the very high percentage of missing values (55% and 70%), using mean, median, or mode imputation would severely distort the distributions and introduce incorrect ranking assumptions. Row deletion is also out of the question as we would lose more than half of our precious dataset. We will fill missing values with a clear category placeholder like `'N/A'` or `'Unranked'` to preserve the rows while maintaining data integrity.

In [ ]:
df_clean = df.copy()
df_clean['Peak'] = df_clean['Peak'].fillna('N/A')
df_clean['All Time Peak'] = df_clean['All Time Peak'].fillna('N/A')
print("Missing Values After Imputation")
print(df_clean[['Peak', 'All Time Peak']].isnull().sum())
print("Preview of Cleaned Dataset")
display(df_clean.head(10))

Missing Values After Imputation
Peak             0
All Time Peak    0
dtype: int64
Preview of Cleaned Dataset


,Rank,Peak,All Time Peak,Actual gross,Adjusted gross (in 2022 dollars),Artist,Tour title,Year(s),Shows,Average gross,Ref.
0,1,1,2,"$780,000,000","$780,000,000",Taylor Swift,The Eras Tour †,2023–2024,56,"$13,928,571",[1]
1,2,1,7[2],"$579,800,000","$579,800,000",Beyoncé,Renaissance World Tour,2023,56,"$10,353,571",[3]
2,3,1[4],2[5],"$411,000,000","$560,622,615",Madonna,Sticky & Sweet Tour ‡[4][a],2008–2009,85,"$4,835,294",[6]
3,4,2[7],10[7],"$397,300,000","$454,751,555",Pink,Beautiful Trauma World Tour,2018–2019,156,"$2,546,795",[7]
4,5,2[4],N/A,"$345,675,146","$402,844,849",Taylor Swift,Reputation Stadium Tour,2018,53,"$6,522,173",[8]
5,6,2[4],10[9],"$305,158,363","$388,978,496",Madonna,The MDNA Tour,2012,88,"$3,467,709",[9]
6,7,2[10],N/A,"$280,000,000","$381,932,682",Celine Dion,Taking Chances World Tour,2008–2009,131,"$2,137,405",[11]
7,7,N/A,N/A,"$257,600,000","$257,600,000",Pink,Summer Carnival †,2023–2024,41,"$6,282,927",[12]
8,9,N/A,N/A,"$256,084,556","$312,258,401",Beyoncé,The Formation World Tour,2016,49,"$5,226,215",[13]
9,10,N/A,N/A,"$250,400,000","$309,141,878",Taylor Swift,The 1989 World Tour,2015,85,"$2,945,882",[14]


In [ ]:
initial_rows = len(df_clean)
duplicate_rows_count = df_clean.duplicated().sum()
df_clean = df_clean.drop_duplicates()
final_rows = len(df_clean)
removed_count = initial_rows - final_rows
print("Duplicate Removal Summary")
print(f"Initial number of rows: {initial_rows}")
print(f"Number of duplicate rows identified: {duplicate_rows_count}")
print(f"Number of duplicate rows removed: {removed_count}")
print(f"Final number of rows: {final_rows}")

Duplicate Removal Summary
Initial number of rows: 20
Number of duplicate rows identified: 0
Number of duplicate rows removed: 0
Final number of rows: 20


In [ ]:
import re
df_standardized = df_clean.copy()
def clean_footnotes(text):
    if pd.isna(text) or not isinstance(text, str):
        return text
    cleaned = re.sub(r'\[.*?\]', '', text)
    cleaned = cleaned.replace('†', '').replace('‡', '').strip()
    return cleaned
def clean_currency(val):
    if pd.isna(val) or not isinstance(val, str):
        return val
    cleaned = val.replace('$', '').replace(',', '').strip()
    try:
        return float(cleaned)
    except ValueError:
        return val
text_cols_to_clean = ['Peak', 'All Time Peak', 'Tour title', 'Ref.']
for col in text_cols_to_clean:
    if col in df_standardized.columns:
        df_standardized[col] = df_standardized[col].apply(clean_footnotes)
currency_cols = ['Actualgross', 'Actualgross', 'Adjustedgross (in 2022 dollars)', 'Average gross']
df_standardized.columns = df_standardized.columns.str.replace('\xa0', ' ')
currency_cols_standardized = ['Actual gross', 'Adjusted gross (in 2022 dollars)', 'Average gross']
for col in currency_cols_standardized:
    if col in df_standardized.columns:
        df_standardized[col] = df_standardized[col].apply(clean_currency)
print("Column Data Types After Standardization")
print(df_standardized.dtypes)
print("Standardized Dataset Preview")
display(df_standardized.head(10))

Column Data Types After Standardization
Rank                                  int64
Peak                                 object
All Time Peak                        object
Actual gross                         object
Adjusted gross (in 2022 dollars)    float64
Artist                               object
Tour title                           object
Year(s)                              object
Shows                                 int64
Average gross                       float64
Ref.                                 object
dtype: object
Standardized Dataset Preview


,Rank,Peak,All Time Peak,Actual gross,Adjusted gross (in 2022 dollars),Artist,Tour title,Year(s),Shows,Average gross,Ref.
0,1,1,2,780000000.0,780000000.0,Taylor Swift,The Eras Tour,2023–2024,56,13928571.0,
1,2,1,7,579800000.0,579800000.0,Beyoncé,Renaissance World Tour,2023,56,10353571.0,
2,3,1,2,411000000.0,560622615.0,Madonna,Sticky & Sweet Tour,2008–2009,85,4835294.0,
3,4,2,10,397300000.0,454751555.0,Pink,Beautiful Trauma World Tour,2018–2019,156,2546795.0,
4,5,2,N/A,345675146.0,402844849.0,Taylor Swift,Reputation Stadium Tour,2018,53,6522173.0,
5,6,2,10,305158363.0,388978496.0,Madonna,The MDNA Tour,2012,88,3467709.0,
6,7,2,N/A,280000000.0,381932682.0,Celine Dion,Taking Chances World Tour,2008–2009,131,2137405.0,
7,7,N/A,N/A,257600000.0,257600000.0,Pink,Summer Carnival,2023–2024,41,6282927.0,
8,9,N/A,N/A,256084556.0,312258401.0,Beyoncé,The Formation World Tour,2016,49,5226215.0,
9,10,N/A,N/A,250400000.0,309141878.0,Taylor Swift,The 1989 World Tour,2015,85,2945882.0,


### Outlier Detection & Handling Strategy (IQR Method)

We will use the **Interquartile Range (IQR)** method to detect outliers in our numeric columns. The IQR method is highly robust and doesn't assume a normal distribution, which is ideal for financial and tour metrics.

#### Columns Analyzed:
1. **`Actual gross`**, **`Adjusted gross (in 2022 dollars)`**, **`Average gross`**:
   - **Strategy:** **Retain but monitor/flag** (or **Cap** to the 1.5x IQR boundary if they skew statistical models).
   - **Justification:** High-grossing concert tours (like Taylor Swift's "The Eras Tour") represent real, highly successful economic events rather than data entry errors. Removing them would strip away the most crucial information in this dataset. However, to prevent them from overly dominating future analysis, we will show where they lie and offer a capped version of the dataset.
2. **`Shows`**:
   - **Strategy:** **Retain/Cap**.
   - **Justification:** Some legacy or farewell tours have an exceptionally high number of shows (e.g., Cher's Living Proof tour with 325 shows). These are valid data points but can be capped at the upper IQR threshold to limit their leverage in regression models.

In [ ]:
import pandas as pd
import numpy as np
df_outliers = df_standardized.copy()
for col in ['Actual gross', 'Adjusted gross (in 2022 dollars)', 'Average gross']:
    if col in df_outliers.columns:
        df_outliers[col] = df_outliers[col].astype(str).str.replace('$', '', regex=False)
        df_outliers[col] = df_outliers[col].str.replace(',', '', regex=False).str.strip()
        df_outliers[col] = pd.to_numeric(df_outliers[col], errors='coerce')
numeric_cols = ['Actual gross', 'Adjusted gross (in 2022 dollars)', 'Average gross', 'Shows']
print("OUTLIER DETECTION USING IQR (1.5x IQR Rule)")
for col in numeric_cols:
    if col in df_outliers.columns:
        col_data = df_outliers[col].dropna()
        if len(col_data) > 0:
            q1 = col_data.quantile(0.25)
            q3 = col_data.quantile(0.75)
            iqr = q3 - q1
            lower_bound = q1 - 1.5 * iqr
            upper_bound = q3 + 1.5 * iqr
            outliers = df_outliers[(df_outliers[col] < lower_bound) | (df_outliers[col] > upper_bound)]
            print(f"Column: '{col}'")
            print(f"IQR Bounds: [{lower_bound:.2f}, {upper_bound:.2f}]")
            print(f"Number of Outliers Detected: {len(outliers)}")
            if not outliers.empty:
                display(outliers[['Artist', 'Tour title', col]])
                df_outliers[col] = np.clip(df_outliers[col], lower_bound, upper_bound)
                print(f"  -> Capped values in '{col}' to the [{lower_bound:.2f}, {upper_bound:.2f}] boundary.")

print("Sample Outlier-Handled Dataset Preview")
display(df_outliers.head(5))

OUTLIER DETECTION USING IQR (1.5x IQR Rule)
Column: 'Actual gross'
IQR Bounds: [-14568925.38, 545614875.62]
Number of Outliers Detected: 2


,Artist,Tour title,Actual gross
0,Taylor Swift,The Eras Tour,780000000.0
1,Beyoncé,Renaissance World Tour,579800000.0


  -> Capped values in 'Actual gross' to the [-14568925.38, 545614875.62] boundary.
Column: 'Adjusted gross (in 2022 dollars)'
IQR Bounds: [25721594.25, 612479178.25]
Number of Outliers Detected: 1


,Artist,Tour title,Adjusted gross (in 2022 dollars)
0,Taylor Swift,The Eras Tour,780000000.0


  -> Capped values in 'Adjusted gross (in 2022 dollars)' to the [25721594.25, 612479178.25] boundary.
Column: 'Average gross'
IQR Bounds: [-3280766.38, 9861298.62]
Number of Outliers Detected: 2


,Artist,Tour title,Average gross
0,Taylor Swift,The Eras Tour,13928571.0
1,Beyoncé,Renaissance World Tour,10353571.0


  -> Capped values in 'Average gross' to the [-3280766.38, 9861298.62] boundary.
Column: 'Shows'
IQR Bounds: [-54.25, 247.75]
Number of Outliers Detected: 1


,Artist,Tour title,Shows
13,Cher,Living Proof: The Farewell Tour,325


  -> Capped values in 'Shows' to the [-54.25, 247.75] boundary.
Sample Outlier-Handled Dataset Preview


,Rank,Peak,All Time Peak,Actual gross,Adjusted gross (in 2022 dollars),Artist,Tour title,Year(s),Shows,Average gross,Ref.
0,1,1,2,5.456149e+08,6.124792e+08,Taylor Swift,The Eras Tour,2023–2024,56.0,9861298.625,
1,2,1,7,5.456149e+08,5.798000e+08,Beyoncé,Renaissance World Tour,2023,56.0,9861298.625,
2,3,1,2,4.110000e+08,5.606226e+08,Madonna,Sticky & Sweet Tour,2008–2009,85.0,4835294.000,
3,4,2,10,3.973000e+08,4.547516e+08,Pink,Beautiful Trauma World Tour,2018–2019,156.0,2546795.000,
4,5,2,N/A,3.456751e+08,4.028448e+08,Taylor Swift,Reputation Stadium Tour,2018,53.0,6522173.000,


In [ ]:
df_dtypes = df_outliers.copy()
df_dtypes['Rank'] = df_dtypes['Rank'].astype(str)
monetary_cols = ['Actual gross', 'Adjusted gross (in 2022 dollars)', 'Average gross']
for col in monetary_cols:
    if col in df_dtypes.columns:
        df_dtypes[col] = pd.to_numeric(df_dtypes[col], errors='coerce').astype(float)
if 'Shows' in df_dtypes.columns:
    df_dtypes['Shows'] = pd.to_numeric(df_dtypes['Shows'], errors='coerce').fillna(0).astype(int)
string_cols = ['Artist', 'Tour title', 'Peak', 'All Time Peak', 'Year(s)', 'Ref.']
for col in string_cols:
    if col in df_dtypes.columns:
        df_dtypes[col] = df_dtypes[col].astype(str).str.strip()
print(" Final Verified Column Data Types")
print(df_dtypes.dtypes)
print("Final Cleaned and Type-Corrected Dataset Sample")
display(df_dtypes.head(5))

 Final Verified Column Data Types
Rank                                 object
Peak                                 object
All Time Peak                        object
Actual gross                        float64
Adjusted gross (in 2022 dollars)    float64
Artist                               object
Tour title                           object
Year(s)                              object
Shows                                 int64
Average gross                       float64
Ref.                                 object
dtype: object
Final Cleaned and Type-Corrected Dataset Sample


,Rank,Peak,All Time Peak,Actual gross,Adjusted gross (in 2022 dollars),Artist,Tour title,Year(s),Shows,Average gross,Ref.
0,1,1,2,5.456149e+08,6.124792e+08,Taylor Swift,The Eras Tour,2023–2024,56,9861298.625,
1,2,1,7,5.456149e+08,5.798000e+08,Beyoncé,Renaissance World Tour,2023,56,9861298.625,
2,3,1,2,4.110000e+08,5.606226e+08,Madonna,Sticky & Sweet Tour,2008–2009,85,4835294.000,
3,4,2,10,3.973000e+08,4.547516e+08,Pink,Beautiful Trauma World Tour,2018–2019,156,2546795.000,
4,5,2,N/A,3.456751e+08,4.028448e+08,Taylor Swift,Reputation Stadium Tour,2018,53,6522173.000,


In [ ]:
import pandas as pd
original_nulls = df.isnull().sum().sum()
original_duplicates = df.duplicated().sum()
original_rows = len(df)
original_dtypes_status = "Inconsistent (monetary as object, hidden characters)"
cleaned_nulls = df_dtypes.isnull().sum().sum()
cleaned_duplicates = df_dtypes.duplicated().sum()
cleaned_rows = len(df_dtypes)
cleaned_dtypes_status = "Correct (monetary as float, rank as string, normalized)"
summary_data = {
    'Metric': [
        'Total Row Count',
        'Total Null Values (Across all columns)',
        'Exact Duplicate Rows Count',
        'Data Type Accuracy Status'
    ],
    'Before Cleaning': [
        original_rows,
        original_nulls,
        original_duplicates,
        original_dtypes_status
    ],
    'After Cleaning': [
        cleaned_rows,
        cleaned_nulls,
        cleaned_duplicates,
        cleaned_dtypes_status
    ]}
comparison_df = pd.DataFrame(summary_data)
print("BEFORE VS. AFTER CLEANING SUMMARY")
display(comparison_df)

BEFORE VS. AFTER CLEANING SUMMARY


,Metric,Before Cleaning,After Cleaning
0,Total Row Count,20,20
1,Total Null Values (Across all columns),25,2
2,Exact Duplicate Rows Count,0,0
3,Data Type Accuracy Status,"Inconsistent (monetary as object, hidden chara...","Correct (monetary as float, rank as string, no..."


In [ ]:
import os
output_file_path = '/content/cleaned_data_quality_report.csv'
df_dtypes.to_csv(output_file_path, index=False)
if os.path.exists(output_file_path):
    print(f"Successfully saved the cleaned dataset to: {output_file_path}")
    print(f"Size: {os.path.getsize(output_file_path)} bytes")
else:
    print("Failed to save the file. Please check the permissions or file path.")

Successfully saved the cleaned dataset to: /content/cleaned_data_quality_report.csv
Size: 1864 bytes
